# <center> التصنيف الائتماني لمؤسسة القروض الصغيرة



## 1. شرح الميزات والبيانات



لدينا بيانات من منظمة القروض الصغيرة. تتضمن البيانات: الطلبات، والحالة (إذا تم قبول الطلب أم لا) ولقطات المحفظة. مستوى المخاطر الحالي (حصة العملاء الافتراضيين) كبير جدًا، ونحن بحاجة إلى تنفيذ بطاقة الأداء الأساسية على الأقل لاستخراج أفضل العملاء.



يمكن العثور على البيانات هنا https://cloud.mail.ru/public/MTPz/zvoEGUjz9



**applications.csv** - يحتوي على معلومات حول التطبيقات
**status.csv** - يحتوي على معلومات إذا تم منح القرض أم لا، ومعلومات قصيرة حول الطلبات السابقة للعميل
**portfolio.csv** - لقطة لكل يوم إذا كان العميل متخلفًا عن السداد أم لا



**applications.csv:**
المدة - مدة القرض
انتهت_التاريخ - حتى ذلك التاريخ يمكن للعميل قبول القرض
القروض_المبلغ - مبلغ القروض (رور) التي حصل عليها العميل
Loans_cnt - عدد القروض التي حصل عليها العميل
Paid_amount - مبلغ القروض (RUR) التي دفعها العميل
Paid_cnt - عدد القروض التي دفعها العميل
ادعاء - العميل لديه ادعاء بشأن القروض السابقة (على سبيل المثال، لم يدفع الغرامات)
location_country - موقع بلد العميل (من عنوان IP)
location_city - موقع مدينة العميل (من عنوان IP)
app_dt - تاريخ تقديم الطلب
verif_data - قام العميل بالتحقق من بياناته
verif_doc - قام العميل بالتحقق من وثيقته
verif_appl - قام العميل بالتحقق من صورته
verif_addres - قام العميل بالتحقق من عنوانه
doc_date - تاريخ إصدار جواز السفر
Client_date - تاريخ التسجيل في مجتمع منظمة القروض الصغيرة
Credit_status - حالة العميل (2: لديه قرض نشط، 1: لديه قروض مغلقة، 0: لم يكن لديه قروض في النظام مطلقًا)
التصنيف، التصنيف 2 - التصنيفات في مجتمع منظمة القروض الصغيرة
المبلغ_المصدر - مبلغ القرض الحاليالمبلغ_الإرجاع - مبلغ القرض الحالي بالإضافة إلى سعر الفائدة
الأصدقاء - الأصدقاء في مجتمع منظمة القروض الصغيرة
app_id - معرف التطبيقات
Client_id - معرف العميل
**status.csv**:
تعليقات - تعليقات الموظفين (عادةً ما تعني شيئًا سيئًا)
القرار - إذا تم منح القرض أم لا (1 - ممنوح)
**portfolio.csv**:
report_dt - تاريخ اللقطة
delq - إذا كان العميل في جنوح أم لا (1 - كان)



## 2. تحليل البيانات الأولية


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack
from datetime import timedelta, date
import datetime
import math
import matplotlib.pyplot as plt
%matplotlib inline


جارٍ تحميل البيانات


In [ ]:
status = pd.read_csv('status.csv')
applications = pd.read_csv('applications.csv')
portfolio = pd.read_csv('portfolio.csv')

In [ ]:
status.head()

In [ ]:
applications.head()

In [ ]:
applications.columns

In [ ]:
portfolio.head()


أولا، نحن بحاجة إلى تحديد المتغير المستهدف. بالطبع، يجب أن لا يتم سداد القرض، ولكن إذا تأخر العميل في السداد لمدة يوم فلا ينبغي أن نعتبره عميلاً متخلفًا عن السداد. ولذلك، نحتاج إلى فحص مقدار أيام التأخير التي سنعتبرها افتراضيًا (الهدف = 1)


In [ ]:
portfolio['report_dt'] = pd.to_datetime(portfolio['report_dt'], format = '%Y-%m-%d')
portfolio['key'] = portfolio['client_id'].apply(lambda x: str(x)) + portfolio['report_dt'].apply(lambda x: str(x))
portfolio['key_last'] = portfolio['client_id'].apply(lambda x: str(x)) + portfolio['report_dt'].apply(lambda x: str(x + timedelta(-1)))

In [ ]:
res = []
d = {}
d[0] = portfolio[(portfolio['delq'] == 0) & (portfolio['report_dt'] < '2018-11-15')]
for i in range(1,30):
    d[i] = portfolio[(portfolio['key_last'].isin(d[i-1]['key'])) & (portfolio['delq'] == 1)]
    res.append(d[i].shape[0])
res = np.array(res)


معدل الاسترداد (كم نسبة العملاء، الذين كانوا في مرحلة التأخر في السداد في يوم ما، ما زالوا في مرحلة التأخر في السداد اعتمادًا على يوم التأخر في السداد):


In [ ]:
plt.plot(res[1:]/res[1], 'bo')
plt.ylim(0,1)
res


سنختار التعريف الافتراضي - أن تكون في حالة تأخر عن السداد لمدة **أكثر من خمسة أيام**، حيث أنه بعد خمسة أيام يتناقص معدل الاسترداد ببطء.



**حساب المتغير المستهدف**



نحن نستبعد جميع الطلبات التي تزيد مدتها عن 60 يومًا، نظرًا لأن الشركة لن تقوم بتقييدها في المستقبل. ونحتاج إلى استبعاد جميع القروض الممنوحة لآخر 60 يومًا + 6 أيام، وإلا فلن نتمكن من حساب المتغير المستهدف


In [ ]:
applications = applications[applications['Term'] <= 60]
applications['app_dt'] = pd.to_datetime(applications['app_dt'], format = '%Y-%m-%d')
applications = applications[applications['app_dt'] <= '2018-10-10']


أضف إشارة تفيد بمنح القرض:


In [ ]:
applications['granted'] = 0
applications.loc[applications['app_id'].isin(status[status['decision'] == 1]['app_id']),'granted'] = 1
applications['granted'].mean()


التحقق من استقرار السعر الممنوح:


In [ ]:
applications['month_gr'] = applications['app_dt'].apply(lambda x: str(x.year*100+x.month))
res = applications[['month_gr', 'granted']].groupby('month_gr').mean().sort_values(by = ['month_gr'])
plt.plot(res)
plt.xticks(rotation='vertical');
plt.ylim((0,1));


هناك انخفاض في يونيو 2018، لكنه بشكل عام مستقر ولا توجد اتجاهات



ترك الطلبات الممنوحة فقط:


In [ ]:
applications = applications[applications['granted'] == 1]


لا توجد بيانات لـ app_id = 12558 في المحفظة، احذف هذا التطبيق (وإلا سيحتوي البرنامج النصي التالي على خطأ):


In [ ]:
applications = applications[applications['app_id'] != 12558]

In [ ]:
%%time
for i in applications['app_id']:
    dt = pd.to_datetime(applications[applications['app_id'] == i]['app_dt'].values[0])
    client_id = applications[applications['app_id'] == i]['client_id'].values[0]
    term = int(applications[applications['app_id'] == i]['Term'].values[0])
    a = portfolio.loc[(portfolio['report_dt'] >= dt + timedelta(term+2)) & (portfolio['report_dt'] <= dt + timedelta(term+6)) & (portfolio['client_id'] == client_id), ['client_id', 'delq']].groupby('client_id').min()
    applications.loc[applications['app_id'] == i, 'target'] = a['delq'].values[0]

In [ ]:
applications['target'].mean()

In [ ]:
res = applications[['month_gr', 'target']].groupby('month_gr').mean().sort_values(by = ['month_gr'])
plt.plot(res)
plt.xticks(rotation='vertical');
plt.ylim((0, 0.3));

المعدل الافتراضي أقل استقرارًا لأنه يمكن لعميل واحد الحصول على أكثر من قرض واحد. والمعدل السيئ يعتمد بقوة على هؤلاء العملاء (إذا كانوا جيدين أو سيئين):


In [ ]:
applications[['client_id', 'app_id']].groupby('client_id').count().reset_index().sort_values(by = 'app_id', ascending = False).head(5)


ولتجنب ذلك، سنترك قرضًا نشطًا واحدًا فقط لكل عميل في كل مرة، على سبيل المثال. إذا حصل العميل على قرض في الأول من يناير لمدة 30 يومًا، فسنستبعد جميع القروض الممنوحة لهذا العميل قبل 31 يناير.


In [ ]:
%%time
client_id = 0
applications = applications.sort_values(by = ['client_id' , 'app_dt', 'app_id'])
applications['sample'] = 0

for i in applications['app_id']:
    if client_id != applications[applications['app_id'] == i]['client_id'].values[0]:
        client_id = applications[applications['app_id'] == i]['client_id'].values[0]
        final_dt = pd.to_datetime(applications[applications['app_id'] == i]['app_dt'].values[0], format = '%Y-%m-%d') + timedelta(int(applications[applications['app_id'] == i]['Term'].values[0]))
        applications.loc[applications['app_id'] == i, 'sample'] = 1    
    else:
        if pd.to_datetime(applications[applications['app_id'] == i]['app_dt'].values[0], format = '%Y-%m-%d') >= final_dt:
            final_dt = pd.to_datetime(applications[applications['app_id'] == i]['app_dt'].values[0], format = '%Y-%m-%d') + timedelta(int(applications[applications['app_id'] == i]['Term'].values[0]))
            applications.loc[applications['app_id'] == i, 'sample'] = 1   

In [ ]:
res = applications[applications['sample'] == 1][['month_gr', 'target']].groupby('month_gr').mean().sort_values(by = ['month_gr'])
plt.plot(res)
plt.xticks(rotation='vertical');
plt.ylim((0, 0.3));


الآن أصبح الأمر أفضل. ولكن يمكننا أن نرى نموًا في صيف 2018. وفي الوقت نفسه شهدنا انخفاضًا في معدل الموافقة في نفس الفترة، وهذا يعني أن تدفق العملاء كان الأسوأ في ذلك الوقت.
الآن ليس لدينا الكثير من البيانات:


In [ ]:
res = applications[applications['sample'] == 1][['month_gr', 'target']].groupby('month_gr').count().sort_values(by = ['month_gr'])
plt.plot(res)
plt.xticks(rotation='vertical');


## 7. المعالجة المسبقة للبيانات



تحقق من نان:


In [ ]:
applications.columns

In [ ]:
applications[['Term', 'expired_date', 'loans_amount', 'loans_cnt',
       'paid_amount', 'paid_cnt', 'pretention', 'location_country',
       'location_city', 'app_dt', 'verif_data', 'verif_doc', 'verif_appl',
       'verif_addres', 'doc_date', 'client_date', 'credit_status', 'rating',
       'rating2', 'amount_issued', 'amount_return', 'friends']].describe()


**الادعاء:**


In [ ]:
applications[['pretention', 'month_gr']].groupby('month_gr').count()


بما أن ثلث المتغير يحتوي على قيم مفقودة، فلن نستخدمه



تحقق من **verif_doc**:


In [ ]:
applications[['verif_doc', 'app_id']].groupby('verif_doc').count()

In [ ]:
applications['verif_doc'] = applications['verif_doc'].fillna(0)


تحقق من **verif_appl**:


In [ ]:
applications[['verif_appl', 'app_id']].groupby('verif_appl').count()

In [ ]:
applications['verif_appl'] = applications['verif_appl'].fillna(0)


تحقق من **location_country** و**location_city**:


In [ ]:
applications['location_country'] = applications['location_country'].fillna('n\a')
applications['location_city'] = applications['location_city'].fillna('n\a')


## 9. إنشاء ميزات جديدة



إضافة ميزات جديدة: الوقت المنقضي منذ تسجيل العميل ومن إصدار المستند (لكل منهما أكثر من أفضل)


In [ ]:
applications['doc_date_i'] = ((pd.to_datetime(applications['app_dt'], format = '%Y-%m-%d')-pd.to_datetime(applications['doc_date'], format = '%Y-%m-%d')).dt.days/365.25).fillna(0)
applications['client_date_i'] = (pd.to_datetime(applications['app_dt'], format = '%Y-%m-%d')-pd.to_datetime(applications['client_date'], format = '%Y-%m-%d')).dt.days/365.25


إضافة ميزة جديدة: متوسط سعر الفائدة (شهرياً). وفقا لعملية القرض الحالية، يقبل العميل الحد الأقصى للسعر الذي يوافق عليه. لن يتم تغيير العملية، وبالتالي يمكننا استخدام هذا المتغير 


In [ ]:
applications['ir'] = (applications['amount_return'] - applications['amount_issued'])/(applications['Term']/30)/applications['amount_issued']

In [ ]:
applications['pti'] = (applications['loans_amount'] - applications['paid_amount'] + applications['amount_return'])/applications['paid_amount']


## 3. تحليل البيانات المرئية الأولية


In [ ]:
appl_short = applications[applications['sample'] == 1]

In [ ]:
appl_short.columns

In [ ]:
var = 'Term'
ratio = 15
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'loans_amount'
ratio = 200000
limit = 5
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'loans_cnt'
ratio = 100
limit = 3
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'rating'
ratio = 100
limit = 3
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'rating2'
ratio = 10
limit = 1
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'amount_issued'
ratio = 5000
limit = 3
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'friends'
ratio = 15
limit = 5
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'client_date_i'
ratio = 3
limit = 5
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'doc_date_i'
ratio = 3
limit = 5
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());

In [ ]:
var = 'ir'
ratio = 0.02
limit = 15
plt.figure(1,figsize=(15,5))
plt.subplot(1,2,1)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (cnt)']).groupby('x').count().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());
plt.subplot(1,2,2)
pd.DataFrame(data = np.concatenate((np.asarray([appl_short[var].apply(lambda x: limit if int(x/ratio) > limit else int(x/ratio))]).T,
                                    appl_short[['target']]), axis = 1), 
             columns = ['x',
                        var + ' (avg target)']).groupby('x').mean().reset_index().sort_values(by = ['x']).plot(x = 'x', 
                                                                                                   kind = 'bar',
                                                                                                    ax=plt.gca());


معدل الفائدة المرتفع جدًا يعني مخاطرة عالية جدًا. يرجع انخفاض المخاطر في المجموعة الأخيرة إلى العملاء ذوي الأجل القصير جدًا (عادةً ما يكون لديهم معدل فائدة أعلى بكثير)



## 5. اختيار المقاييس


نحن بحاجة إلى تقسيم العملاء إلى "الجيدين" و"السيئين". لكننا لا نحتاج إلى فصل العملاء "السيئين" عن العملاء "السيئين جدًا". لذلك، أولاً، نقوم برفض العملاء السيئين جدًا - بمعدل فائدة مرتفع جدًا.
بعد ذلك، سنستخدم المقياس القياسي لتسجيل نقاط البنك - roc-auc.


In [ ]:
df = appl_short[['target', 'ir', 'Term']]
df['Term_gr'] = df['Term'].apply(lambda x: 7 if x <= 7 else 29 if x <= 29 else 30 if x == 30 else 60)
df['ir_gr'] = df['ir'].apply(lambda x: 15 if int(x/0.02) > 15 else int(x/0.02))
df['cnt'] = 1
df2 = df.groupby(['Term_gr', 'ir_gr']).sum().reset_index()
df2['avg_target'] = df2['target']/df2['cnt']

In [ ]:
df2[['Term_gr', 'ir_gr', 'avg_target', 'cnt']]

In [ ]:
df = appl_short[['target', 'ir', 'Term']]
df['Term_gr'] = df['Term'].apply(lambda x: 7 if x <= 7 else 29 if x <= 29 else 30 if x == 30 else 60)
df['ir_gr'] = df['ir'].apply(lambda x: 11 if int(x/0.02) > 11 else 7 if int(x/0.02) >= 7 else 6)
df['cnt'] = 1
df2 = df.groupby(['Term_gr', 'ir_gr']).sum().reset_index()
df2['avg_target'] = df2['target']/df2['cnt']
df2[['Term_gr', 'ir_gr', 'avg_target', 'cnt']]


وأخيرا، سوف نقوم برفض العملاء الذين لديهم 
    - سعر الفائدة >= 0.13 إذا كانت المدة >= 30
    - سعر الفائدة >= 0.21 إذا كانت المدة > 7


In [ ]:
applications['auto_decl'] = 0
applications.loc[(applications['ir'] >= 0.13) & (applications['Term'] >= 30),'auto_decl'] = 1
applications.loc[(applications['ir'] >= 0.21) & (applications['Term'] > 7),'auto_decl'] = 1
appl_short = applications[(applications['sample'] == 1) & (applications['auto_decl'] == 0)]

In [ ]:
applications[(applications['sample'] == 1) & (applications['auto_decl'] == 0)]['target'].mean(), \
applications[(applications['sample'] == 1) & (applications['auto_decl'] == 1)]['target'].mean(), \
applications[(applications['sample'] == 1) & (applications['auto_decl'] == 0)]['target'].count(), \
applications[(applications['sample'] == 1) & (applications['auto_decl'] == 1)]['target'].count()


نرفض 124 ملاحظة بنسبة خطورة تزيد عن 30%



## 6. اختيار النموذج



سنختار نموذج catboost لأنه يعمل بشكل جيد جدًا مع المتغيرات الفئوية. 


In [ ]:
appl_short = appl_short.sort_values(by = ['app_dt', 'app_id'])
X_train = appl_short[['location_country','location_city', 'verif_data', 'verif_doc', 'verif_appl',
                       'verif_addres', 'credit_status', 'Term', 'rating', 'pti',
                   'rating2', 'amount_issued', 'amount_return', 'friends', 'doc_date_i','client_date_i', 'ir']].values
y_train = appl_short['target'].values

In [ ]:
X_train_part, X_valid, y_train_part, y_valid = \
    train_test_split(X_train, y_train, 
                     test_size=0.3, random_state=17)

In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(random_seed = 17, thread_count = 4, verbose = False)
model.fit(X_train_part, y_train_part, cat_features = [0,1,2,3,4,5,6])

In [ ]:
cb_train_pred = model.predict_proba(X_valid)[:, 1]
roc_auc_score(y_valid, cb_train_pred)


## 8. التحقق من الصحة وتعديل المعلمات الفائقة للنموذج


In [ ]:
X_train = appl_short[['location_country','location_city', 'verif_data', 'verif_doc', 'verif_appl',
                       'verif_addres', 'credit_status', 'Term', 'rating', 'pti',
                   'rating2', 'amount_issued', 'amount_return', 'friends', 'doc_date_i','client_date_i', 'ir']].values
y_train = appl_short['target'].values


سنجد أفضل قيم max_Deep وn_estimators، وسنفترض أن المعلمة cv = 3


In [ ]:
%%time
res = []
cv = 3

step = int(X_train.shape[0]/(cv + 1))

for max_depth in range(2,7,2):
    for n_estimators in range(500, 2000, 500):            
        for i in range(1,cv+1):
            X_train_part = X_train[:step*i-1,:]
            y_train_part = y_train[:step*i-1]
            if i < cv:
                X_valid = X_train[step*i:step*(i+1),:]
                y_valid = y_train[step*i:step*(i+1)]
            else:
                X_valid = X_train[step*i:,:]
                y_valid = y_train[step*i:]

            model = CatBoostClassifier(random_seed = 17, thread_count = 4, verbose = False,
                                       max_depth = max_depth, n_estimators = n_estimators)
            model.fit(X_train_part, y_train_part, cat_features = [0,1,2,3,4,5,6])
            
            cb_train_pred = model.predict_proba(X_train_part)[:, 1]
            train_res = roc_auc_score(y_train_part, cb_train_pred)
            
            cb_valid_pred = model.predict_proba(X_valid)[:, 1]
            cv_res = roc_auc_score(y_valid, cb_valid_pred)
            res.append([max_depth, n_estimators, i, train_res, cv_res])
            print(max_depth, n_estimators, i, train_res, cv_res)


In [ ]:
res = pd.DataFrame(data = res, columns = ['max_depth', 'n_estimators', 'cv_iter', 'train_res', 'cv_res'])
res[['max_depth', 'n_estimators', 'train_res', 'cv_res']].groupby(['max_depth', 'n_estimators']).\
    mean().reset_index().sort_values(by = ['cv_res'], ascending = False)


## 10. رسم منحنيات التدريب والتحقق من الصحة



أقصى عمق:


In [ ]:
%%time
res = []
cv = 3

step = int(X_train.shape[0]/(cv + 1))

n_estimators = 100

for max_depth in range(2,11):
    for i in range(1,cv+1):
        X_train_part = X_train[:step*i-1,:]
        y_train_part = y_train[:step*i-1]
        if i < cv:
            X_valid = X_train[step*i:step*(i+1),:]
            y_valid = y_train[step*i:step*(i+1)]
        else:
            X_valid = X_train[step*i:,:]
            y_valid = y_train[step*i:]

        model = CatBoostClassifier(random_seed = 17, thread_count = 4, verbose = False,
                                   max_depth = max_depth, n_estimators = n_estimators)
        model.fit(X_train_part, y_train_part, cat_features = [0,1,2,3,4,5,6])

        cb_train_pred = model.predict_proba(X_train_part)[:, 1]
        train_res = roc_auc_score(y_train_part, cb_train_pred)

        cb_valid_pred = model.predict_proba(X_valid)[:, 1]
        cv_res = roc_auc_score(y_valid, cb_valid_pred)
        res.append([max_depth, n_estimators, i, train_res, cv_res])

df_res = pd.DataFrame(data = res, columns = ['max_depth', 'n_estimators', 'cv_iter', 'train_res', 'cv_res'])
x = np.linspace(2,10,9)
y1 = df_res[['max_depth', 'n_estimators', 'train_res']].groupby(['max_depth', 'n_estimators']).\
        mean()['train_res'].values
y2 = df_res[['max_depth', 'n_estimators', 'cv_res']].groupby(['max_depth', 'n_estimators']).\
        mean().reset_index()['cv_res'].values
plt.plot(x, y1, '-b', label = 'train')
plt.plot(x, y2, '-r', label = 'valid')
plt.legend(loc='upper left')


عدد المقدرين:


In [ ]:
%%time
res = []
cv = 3

step = int(X_train.shape[0]/(cv + 1))

max_depth = 2

for n_estimators in range(100,1500,100):
    for i in range(1,cv+1):
        X_train_part = X_train[:step*i-1,:]
        y_train_part = y_train[:step*i-1]
        if i < cv:
            X_valid = X_train[step*i:step*(i+1),:]
            y_valid = y_train[step*i:step*(i+1)]
        else:
            X_valid = X_train[step*i:,:]
            y_valid = y_train[step*i:]

        model = CatBoostClassifier(random_seed = 17, thread_count = 4, verbose = False,
                                   max_depth = max_depth, n_estimators = n_estimators)
        model.fit(X_train_part, y_train_part, cat_features = [0,1,2,3,4,5,6])

        cb_train_pred = model.predict_proba(X_train_part)[:, 1]
        train_res = roc_auc_score(y_train_part, cb_train_pred)

        cb_valid_pred = model.predict_proba(X_valid)[:, 1]
        cv_res = roc_auc_score(y_valid, cb_valid_pred)
        res.append([max_depth, n_estimators, i, train_res, cv_res])
        
df_res = pd.DataFrame(data = res, columns = ['max_depth', 'n_estimators', 'cv_iter', 'train_res', 'cv_res'])
x = np.linspace(100,1400,14)
y1 = df_res[['max_depth', 'n_estimators', 'train_res']].groupby(['max_depth', 'n_estimators']).\
        mean()['train_res'].values
y2 = df_res[['max_depth', 'n_estimators', 'cv_res']].groupby(['max_depth', 'n_estimators']).\
        mean().reset_index()['cv_res'].values
plt.plot(x, y1, '-b', label = 'train')
plt.plot(x, y2, '-r', label = 'valid')
plt.legend(loc='upper left')


## 11. التنبؤ بالعينات الاختبارية أو المحتجزة



**النموذج النهائي:**


In [ ]:
X_train = appl_short[['location_country','location_city', 'verif_data', 'verif_doc', 'verif_appl',
                       'verif_addres', 'credit_status', 'Term', 'rating', 'pti',
                   'rating2', 'amount_issued', 'amount_return', 'friends', 'doc_date_i','client_date_i', 'ir']].values
y_train = appl_short['target'].values

X_train_part, X_valid, y_train_part, y_valid = \
    train_test_split(X_train, y_train, 
                     test_size=0.3, random_state=17)
model = CatBoostClassifier(random_seed = 17, thread_count = 4, verbose = False,
                            max_depth = 2, n_estimators = 1000)
model.fit(X_train_part, y_train_part, cat_features = [0,1,2,3,4,5,6])
cb_train_pred = model.predict_proba(X_valid)[:, 1]
roc_auc_score(y_valid, cb_train_pred)


هذه النتيجة أقل مما لدينا في التكرار الأول للنموذج، ولكن يبدو أنها أكثر استقرارًا.



## 12. الاستنتاجات



أولا، نحتاج إلى تحديد الحد الأقصى، لأن الهدف الرئيسي لهذا المشروع هو تقدير الحجم المحتمل للقروض. لإعداد الحد الفاصل، نحتاج إلى اختيار قيمة النتيجة وفقًا لمستوى المخاطرة. مستوى المخاطرة المقبول هو 2% (يتم حسابه من الهامش)


In [ ]:
res = []
res = np.concatenate((np.asarray([cb_train_pred]).T, np.asarray([y_valid]).T), axis = 1)
df_res = pd.DataFrame(data = res, columns = ['score', 'bad'])
df_res = df_res.sort_values(by = ['score'])
df_res['score_gr'] = df_res['score'].apply(lambda x: 'good' if x <= 0.027 else 'bad')
print(df_res[['score_gr', 'bad']].groupby('score_gr').mean(), df_res[['score_gr', 'bad']].groupby('score_gr').count())


حصة العملاء الجيدين صغيرة جدًا



**الخلاصة:** 
1. من غير الممكن بناء النموذج على بيانات التطبيق فقط، ويجب استخدام بيانات إضافية
2. لتضمين متغير نموذج التاريخ الائتماني (الآن غير متوفر في عملية الائتمان)